# 🧩 Module 08 — Functions

## 🎯 What you will learn
1. Defining and calling functions; `return` vs `print`
2. Parameters vs arguments: positional, keyword, default, `*args`, `**kwargs`, positional-only / keyword-only
3. Scope: LEGB rule, `global`, `nonlocal`
4. How Python passes arguments ("pass by object reference")
5. The mutable default argument trap
6. Docstrings, type hints, first-class functions
7. Lambda, `map`, `filter`, `reduce`, `sorted(key=)`
8. Recursion basics, closures
9. Hidden facts & tricks

**Exam relevance:** ⭐⭐⭐⭐⭐

---
## 1. Why functions?
A function is a **named, reusable block of code**. Benefits: reuse (DRY), organisation, testing, abstraction.

In [1]:
def greet(name):
    """Return a greeting for `name`."""
    return f"Hello, {name}!"

message = greet("Asha")       # call
print(message)
print(greet("Ravi"))

Hello, Asha!
Hello, Ravi!


Anatomy:
```
def function_name(parameters):     ← header ends with a colon
    """docstring"""                 ← optional documentation
    body (indented)
    return value                    ← optional; default return is None
```

### `return` vs `print`
- `print` shows something on screen; the function's result is `None`.
- `return` **hands a value back** to the caller so it can be stored/used. It also immediately ends the function.

In [2]:
def add_print(a, b):
    print(a + b)

def add_return(a, b):
    return a + b

x = add_print(2, 3)
y = add_return(2, 3)
print(x, y)                   # None 5
print(add_return(add_return(1, 2), 3))

def early(n):
    if n < 0:
        return "negative"     # exits immediately
    return "non-negative"
print(early(-1), early(1))

def multiple():
    return 1, "two", [3]      # actually returns ONE tuple
a, b, c = multiple()
print(multiple(), a, b, c)

5
None 5
6
negative non-negative
(1, 'two', [3]) 1 two [3]


## 2. Parameters & arguments

- **Parameter** = variable in the definition. **Argument** = actual value passed in the call.

### 2.1 Positional & keyword arguments

In [3]:
def describe(name, age, city):
    return f"{name}, {age}, from {city}"

print(describe("Asha", 19, "Pune"))                      # positional (order matters)
print(describe(city="Pune", name="Asha", age=19))        # keyword (order free)
print(describe("Asha", city="Pune", age=19))             # mix: positional first

Asha, 19, from Pune
Asha, 19, from Pune
Asha, 19, from Pune


### 2.2 Default values

In [4]:
def power(base, exp=2):
    return base ** exp

print(power(5), power(5, 3), power(exp=4, base=2))

def connect(host="localhost", port=8080, secure=False):
    return f"{'https' if secure else 'http'}://{host}:{port}"
print(connect(), connect(port=9000), connect("example.com", secure=True))

# Non-default parameters can't follow defaults:
try:
    exec("def bad(a=1, b): pass")
except SyntaxError as e:
    print("SyntaxError:", e.msg)

25 125 16
http://localhost:8080 http://localhost:9000 https://example.com:8080
SyntaxError: parameter without a default follows parameter with a default


### 2.3 ⚠️ The mutable default argument trap (famous!)
Default values are evaluated **once, when the function is defined**, not on each call.

In [5]:
def add_item(item, box=[]):          # ❌ BAD
    box.append(item)
    return box
print(add_item(1))
print(add_item(2))                   # [1, 2] -- surprise! same list reused

def add_item_ok(item, box=None):     # ✅ GOOD pattern
    if box is None:
        box = []
    box.append(item)
    return box
print(add_item_ok(1), add_item_ok(2))

print(add_item.__defaults__)          # the hidden shared list lives here

[1]
[1, 2]
[1] [2]
([1, 2],)


### 2.4 `*args` — variable number of positional arguments (collected into a **tuple**)

In [6]:
def total(*numbers):
    print(type(numbers), numbers)
    return sum(numbers)

print(total(1, 2, 3))
print(total())
print(total(*[10, 20, 30]))          # * unpacks a list into arguments at the call

<class 'tuple'> (1, 2, 3)
6
<class 'tuple'> ()
0
<class 'tuple'> (10, 20, 30)
60


### 2.5 `**kwargs` — variable number of keyword arguments (collected into a **dict**)

In [7]:
def profile(**info):
    print(type(info))
    for k, v in info.items():
        print(f"  {k}: {v}")

profile(name="Asha", age=19, city="Pune")
profile(**{"a": 1, "b": 2})          # ** unpacks a dict into keyword args

def everything(a, b=2, *args, key="k", **kwargs):
    return a, b, args, key, kwargs
print(everything(1))
print(everything(1, 9, 8, 7, key="z", extra=True))

<class 'dict'>
  name: Asha
  age: 19
  city: Pune
<class 'dict'>
  a: 1
  b: 2
(1, 2, (), 'k', {})
(1, 9, (8, 7), 'z', {'extra': True})


### 2.6 Order of parameters (memorise)
```
def f(positional, /, normal, default=1, *args, keyword_only, **kwargs)
```
1. positional-only (before `/`) — Python 3.8+
2. normal positional-or-keyword
3. defaults
4. `*args`
5. keyword-only (after `*` or `*args`)
6. `**kwargs`

In [8]:
def f(a, b, /, c, *, d):
    return a, b, c, d

print(f(1, 2, 3, d=4))
print(f(1, 2, c=3, d=4))
for bad in ["f(a=1, b=2, c=3, d=4)", "f(1, 2, 3, 4)"]:
    try:
        eval(bad)
    except TypeError as e:
        print("TypeError:", e)

(1, 2, 3, 4)
(1, 2, 3, 4)
TypeError: f() got some positional-only arguments passed as keyword arguments: 'a, b'
TypeError: f() takes 3 positional arguments but 4 were given


## 3. Scope — the LEGB rule

Python looks up a name in this order: **L**ocal → **E**nclosing → **G**lobal → **B**uilt-in.

In [9]:
x = "global"

def outer():
    x = "enclosing"
    def inner():
        x = "local"
        print("inner sees:", x)
    inner()
    print("outer sees:", x)

outer()
print("module sees:", x)
print(len)                              # Built-in scope

inner sees: local
outer sees: enclosing
module sees: global
<built-in function len>


### `global` and `nonlocal`

In [10]:
counter = 0
def increment():
    global counter               # without this: UnboundLocalError
    counter += 1
increment(); increment()
print(counter)

def make_counter():
    count = 0
    def inc():
        nonlocal count           # refers to the enclosing function's variable
        count += 1
        return count
    return inc

c1 = make_counter()
print(c1(), c1(), c1())
c2 = make_counter()              # independent state
print(c2())

# Reading a global is fine; ASSIGNING makes it local unless declared global
y = 10
def read_y():  return y
def bad_y():
    try:
        y += 1
    except UnboundLocalError as e:
        return f"UnboundLocalError: {e}"
print(read_y(), bad_y())

2
1 2 3
1
10 UnboundLocalError: cannot access local variable 'y' where it is not associated with a value


## 4. How arguments are passed: "pass by object reference"
Python passes a **reference to the object**. Rebinding the parameter name doesn't affect the caller, but *mutating* a mutable object does.

In [11]:
def modify(lst, num, s):
    lst.append(99)         # mutates caller's list  -> visible
    num += 1               # rebinds local name    -> invisible (ints immutable)
    s += "!"               # new string, rebinding  -> invisible

L, N, S = [1], 10, "hi"
modify(L, N, S)
print(L, N, S)

def reassign(lst):
    lst = [0, 0]           # rebinding: caller unaffected
reassign(L); print(L)

[1, 99] 10 hi
[1, 99]


## 5. Functions are first-class objects
Functions can be assigned to variables, stored in lists/dicts, passed as arguments and returned from other functions.

In [12]:
def square(x): return x * x
def cube(x): return x ** 3

f = square
print(f(4), f.__name__)

funcs = {"sq": square, "cu": cube}
for name, fn in funcs.items():
    print(name, fn(3))

def apply_twice(fn, value):
    return fn(fn(value))
print(apply_twice(square, 3))          # (3²)² = 81

def make_multiplier(n):
    def multiplier(x):
        return x * n
    return multiplier
double = make_multiplier(2)
triple = make_multiplier(3)
print(double(5), triple(5))

16 square
sq 9
cu 27
81
10 15


## 6. Lambda (anonymous) functions
`lambda args: expression` — a tiny one-expression function with no name and no statements.

In [13]:
add = lambda a, b: a + b
print(add(2, 3))
print((lambda x: x * 2)(21))
print(sorted(["pear", "fig", "apple"], key=lambda w: len(w)))
print(sorted([(1, "b"), (2, "a")], key=lambda t: t[1]))
is_even = lambda n: n % 2 == 0
print(is_even(4))

5
42
['fig', 'pear', 'apple']
[(2, 'a'), (1, 'b')]
True


### `map`, `filter`, `reduce`

In [14]:
from functools import reduce
nums = [1, 2, 3, 4, 5, 6]

print(list(map(lambda x: x ** 2, nums)))               # transform each
print(list(filter(lambda x: x % 2 == 0, nums)))        # keep if True
print(reduce(lambda acc, x: acc + x, nums))            # fold to one value: 21
print(reduce(lambda a, b: a * b, nums, 1))             # product = 720
print(list(map(lambda a, b: a + b, [1, 2], [10, 20])))   # multiple iterables
print(list(map(str.upper, ["a", "b"])))

# Comprehensions are usually more readable:
print([x ** 2 for x in nums], [x for x in nums if x % 2 == 0])

[1, 4, 9, 16, 25, 36]
[2, 4, 6]
21
720
[11, 22]
['A', 'B']
[1, 4, 9, 16, 25, 36] [2, 4, 6]


## 7. Docstrings & type hints

In [15]:
def area_of_circle(radius: float, precision: int = 2) -> float:
    """
    Return the area of a circle.

    Args:
        radius: radius of the circle (>= 0)
        precision: decimal places to round to

    Returns:
        The area as a float.

    Raises:
        ValueError: if radius is negative.
    """
    if radius < 0:
        raise ValueError("radius must be non-negative")
    import math
    return round(math.pi * radius ** 2, precision)

print(area_of_circle(3))
print(area_of_circle.__doc__.splitlines()[1].strip())
print(area_of_circle.__annotations__)

28.27
Return the area of a circle.
{'radius': <class 'float'>, 'precision': <class 'int'>, 'return': <class 'float'>}


> Type hints are **not enforced** at runtime; they document intent and power tools like `mypy` and IDE autocomplete.

## 8. Recursion (preview — full module later)
A function that calls itself. Needs a **base case** and progress toward it.

In [16]:
def factorial(n):
    if n <= 1:                 # base case
        return 1
    return n * factorial(n - 1)   # recursive case

def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(factorial(6), [fib(i) for i in range(10)])

import sys
print("recursion limit:", sys.getrecursionlimit())

720 [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
recursion limit: 1000


## 9. Useful built-in functions (know them all)

In [17]:
nums = [3, -1, 4, -1, 5]
print(abs(-7), pow(2, 5), divmod(17, 5), round(3.14159, 3))
print(len(nums), sum(nums), min(nums), max(nums), sorted(nums))
print(list(zip("ab", [1, 2])), list(enumerate("ab")), list(reversed(nums)))
print(all([1, 1]), any([0, 0]), bool(0))
print(isinstance(3, int), type(3), id(3) == id(3))
print(chr(97), ord("a"), bin(5), hex(255), oct(8))
print(list(range(3)), list(map(int, "123")), callable(len))
print(eval("2 + 3 * 4"), repr("hi"), str("hi"))
print(getattr("abc", "upper")(), hasattr([], "append"))
print(vars().keys() is not None, "print" in dir(__builtins__))

7 32 (3, 2) 3.142
5 10 -1 5 [-1, -1, 3, 4, 5]
[('a', 1), ('b', 2)] [(0, 'a'), (1, 'b')] [5, -1, 4, -1, 3]
True False False
True <class 'int'> True
a 97 0b101 0xff 0o10
[0, 1, 2] [1, 2, 3] True
14 'hi' hi
ABC True
True False


## 10. Practice-style programs

In [18]:
def is_prime(n):
    if n < 2: return False
    return all(n % d for d in range(2, int(n ** 0.5) + 1))

def primes_upto(n):
    return [i for i in range(2, n + 1) if is_prime(i)]

def gcd(a, b):
    return a if b == 0 else gcd(b, a % b)

def lcm(a, b):
    return a * b // gcd(a, b)

def is_palindrome(s):
    s = "".join(c.lower() for c in s if c.isalnum())
    return s == s[::-1]

def stats(*nums):
    n = len(nums)
    mean = sum(nums) / n
    nums = sorted(nums)
    median = nums[n // 2] if n % 2 else (nums[n // 2 - 1] + nums[n // 2]) / 2
    return {"mean": mean, "median": median, "range": nums[-1] - nums[0]}

def compose(*fns):
    def run(x):
        for fn in reversed(fns):
            x = fn(x)
        return x
    return run

print(primes_upto(30), gcd(48, 18), lcm(4, 6))
print(is_palindrome("Was it a car or a cat I saw?"))
print(stats(4, 1, 7, 2))
inc_then_double = compose(lambda x: x * 2, lambda x: x + 1)
print(inc_then_double(5))    # (5+1)*2 = 12

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29] 6 12
True
{'mean': 3.5, 'median': 3.0, 'range': 6}
12


---
# 🕵️ Hidden Facts & Pro Tricks

In [19]:
# 1. Functions have attributes (they are objects)
def hello(): pass
hello.calls = 0
hello.calls += 1
print(hello.calls, hello.__name__, hello.__module__)

# 2. Introspection with inspect
import inspect
def demo(a, b=2, *args, k=None, **kw): pass
print(inspect.signature(demo))

# 3. Memoisation for free: functools.cache / lru_cache
from functools import cache
@cache
def fib_fast(n):
    return n if n < 2 else fib_fast(n - 1) + fib_fast(n - 2)
print(fib_fast(300))                 # instant! naive recursion would take ages
print(fib_fast.cache_info())

# 4. partial: pre-fill some arguments
from functools import partial
int_from_binary = partial(int, base=2)
print(int_from_binary("1011"))

# 5. Unpack in the call and return several things
print(*sorted("hello"), sep="")

# 6. Function returning functions — currying
adder = lambda a: lambda b: a + b
print(adder(3)(4))

# 7. Keyword-only args make APIs clearer
def create_user(name, *, admin=False):
    return name, admin
print(create_user("x", admin=True))

# 8. Default arg evaluated at def time can be used deliberately (fast local binding)
import math
def fast_sqrt(x, _sqrt=math.sqrt):
    return _sqrt(x)
print(fast_sqrt(16))

# 9. Callable objects: any class with __call__ acts like a function
class Adder:
    def __init__(self, n): self.n = n
    def __call__(self, x): return x + self.n
print(Adder(10)(5), callable(Adder(1)))

# 10. sorted(key=) with multiple criteria and itemgetter/attrgetter (fast, C-level)
from operator import itemgetter
rows = [("b", 2), ("a", 2), ("c", 1)]
print(sorted(rows, key=itemgetter(1, 0)))

# 11. Every function returns something: None by default
print(print("x"))

# 12. Late binding in closures — a classic gotcha and its fix
bad = [lambda: i for i in range(3)]
good = [lambda i=i: i for i in range(3)]
print([f() for f in bad], [f() for f in good])

# 13. one-line swap function, and `*` in the middle of the arguments
def swap(a, b): return b, a
print(swap(1, 2))

# 14. dis.dis shows the bytecode of a function; __code__ has co_varnames etc.
print(demo.__code__.co_varnames)

1 hello __main__
(a, b=2, *args, k=None, **kw)
222232244629420445529739893461909967206666939096499764990979600
CacheInfo(hits=298, misses=301, maxsize=None, currsize=301)
11
ehllo
7
('x', True)
4.0
15 True
[('c', 1), ('a', 2), ('b', 2)]
x
None
[2, 2, 2] [0, 1, 2]
(2, 1)
('a', 'b', 'k', 'args', 'kw')


**Hidden fact:** Python has **no function overloading** — defining a function with the same name twice just replaces the first. Use default args, `*args`, or `functools.singledispatch`.

**Hidden fact:** recursion depth defaults to 1000 (`sys.setrecursionlimit` changes it, but deep recursion should become a loop). CPython does **not** optimise tail calls.

**Hidden fact:** Python does not have "pass by value" or "pass by reference" — it is **pass by assignment**: the parameter becomes another name for the same object.

---
# 📝 Exam Corner

**Theory**
1. What is a function? Advantages of using functions.
2. Types of arguments (positional, keyword, default, variable-length) with examples.
3. Difference between parameter & argument; `return` & `print`.
4. Explain scope and the LEGB rule; `global` vs `nonlocal`.
5. What is a lambda function? Difference between lambda and `def`.
6. Explain `map`, `filter`, `reduce`.
7. What is recursion? Advantages/disadvantages.
8. Explain `*args` and `**kwargs`.

**Output prediction**

In [20]:
def f(a, b=[]):
    b.append(a)
    return b

print(f(1), f(2), f(3, []))

x = 5
def g():
    x = 10
    return x
print(g(), x)

def h(*args, **kwargs):
    return len(args), len(kwargs)
print(h(1, 2, a=3), h(), h(*[1, 2, 3], **{"z": 1}))

print((lambda x, y=2: x ** y)(3))
print(list(map(lambda x: x + 1, filter(lambda x: x > 1, [1, 2, 3]))))

[1, 2] [1, 2] [3]
10 5
(2, 1) (0, 0) (3, 1)
9
[3, 4]


<details><summary>💡 Answers</summary>

```
[1, 2] [1, 2] [3]
10 5
(2, 1) (0, 0) (3, 1)
9
[3, 4]
```
Careful: `print(f(1), f(2), f(3, []))` evaluates all three calls **before** printing. The first two return the *same* list object, which by then holds `[1, 2]` → both show `[1, 2]`.
</details>

---
## ✅ Summary
- `def name(params): ... return value`. No `return` → `None`.
- Argument kinds: positional, keyword, default, `*args` (tuple), `**kwargs` (dict).
- LEGB scoping; use `global`/`nonlocal` sparingly.
- Mutable defaults are shared → use `None`.
- Functions are first-class: pass, return, store them. Lambdas for tiny throwaway functions.

➡️ **Next:** Module 09 — Comprehensions, Iterators, Generators & Decorators.